In [1]:
import rclpy

from turtlebot4_navigation.turtlebot4_navigator import TurtleBot4Directions, TurtleBot4Navigator

In [2]:
# from optparse import OptionParser
# import inspect

# inspect.getmembers(navigator, predicate=inspect.ismethod)

In [3]:
# navigator.stampPose()


# Check Docking functions

In [2]:
rclpy.init()

In [3]:
navigator = TurtleBot4Navigator()

In [4]:
navigator.getDockedStatus() 

[WARN] [1770037101.196802929] [basic_navigator]: New publisher discovered on topic 'hazard_detection', offering incompatible QoS. No messages will be received from it. Last incompatible policy: RELIABILITY


False

In [10]:
navigator.undock()

[INFO] [1770037281.868732398] [basic_navigator]: Undocking...
[INFO] [1770037287.507483102] [basic_navigator]: Undock succeeded


In [6]:
navigator.getDockedStatus()

False

In [11]:
navigator.dock()

[INFO] [1770038339.895752565] [basic_navigator]: Docking...
[INFO] [1770038360.449461250] [basic_navigator]: Dock succeeded


In [11]:
navigator.getDockedStatus()

True

In [12]:
rclpy.shutdown()

# Test

In [1]:
import rclpy

from turtlebot4_navigation.turtlebot4_navigator import TurtleBot4Directions, TurtleBot4Navigator

In [2]:
from rclpy.node import Node 
from nav_msgs.msg import Odometry
from transformations import euler_from_quaternion
from rclpy.qos import QoSProfile, ReliabilityPolicy

import math


class MyNode(Node):
    def __init__(self):
        super().__init__("py_test")
        self.qos = QoSProfile(
            depth=10,
            reliability=ReliabilityPolicy.BEST_EFFORT
        )
        self.sub = self.create_subscription(Odometry, '/odom', self.odom_cb, self.qos)

    def odom_cb(self, msg):
        self.current_odom_pose = msg.pose.pose
        print("X")

    def get_pose_with_distance(self, d):
        q = self.current_odom_pose.orientation
        yaw = euler_from_quaternion([q.x, q.y, q.z, q.w])[0]

        x = self.current_odom_pose.position.x + math.cos(yaw) * d
        
        y = self.current_odom_pose.position.y + math.sin(yaw) * d

        return (x, y, yaw)


rclpy.init()
node = MyNode()


In [19]:
navigator = TurtleBot4Navigator()
initial_pose = navigator.getPoseStamped([0.0, 0.0], TurtleBot4Directions.NORTH)


[WARN] [1768827983.995941787] [rcl.logging_rosout]: Publisher already registered for provided node name. If this is due to multiple nodes with the same name then all logs for that logger name will go out over the existing publisher. As soon as any node with that name is destructed it will unregister the publisher, preventing any further logs for that name from being published on the rosout topic.


In [38]:
for _ in range(10):
    rclpy.spin_once(node)

pose = node.get_pose_with_distance(1)

# (-3.448481465520487, 0.7228716572887234, -3.133012336584858)
pose

X
X
X
X
X
X


(-1.079250010224082, 0.5111932276205936, -0.8491530224465729)

In [39]:
stamped_pose = navigator.getPoseStamped(pose[:2], pose[-1])
stamped_pose

geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1768828632, nanosec=300839112), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=-1.079250010224082, y=0.5111932276205936, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=-0.007410190229075013, w=0.999972544163473)))

## Basic navigation

In [40]:
navigator.goToPose(stamped_pose)

[INFO] [1768828633.267400424] [basic_navigator]: Navigating to goal: -1.079250010224082 0.5111932276205936...


True

In [41]:
navigator.goToPose(initial_pose)

[INFO] [1768828687.171733538] [basic_navigator]: Navigating to goal: 0.0 0.0...


True

In [37]:
navigator.spin(spin_dist=1.34)

[INFO] [1768828628.213524118] [basic_navigator]: Spinning to angle 1.34....


True

In [ ]:
navigator.spin(spin_dist=0.57, time_allowance=10)

In [ ]:
navigator.driveOnHeading(dist=0.15, speed=0.025, time_allowance=10, disable_collision_checks=False)

# Check Navigation functions

## Init rclpy and create Turtlebot4Navigator object

In [2]:
rclpy.init()
navigator = TurtleBot4Navigator()

## Set initial position and orientation

<img src="../maps/test-arena-2.png" alt="isolated" width="400"/>

In [3]:
# Start on dock
if not navigator.getDockedStatus():
    navigator.info('Docking before intialising pose')
    navigator.dock()
    
# Set initial pose
initial_pose = navigator.getPoseStamped([0.0, 0.0], TurtleBot4Directions.NORTH)
# initial_pose = navigator.getPoseStamped([-0.1, -0.1], 2.5)
navigator.setInitialPose(initial_pose)


[WARN] [1768831997.000746590] [basic_navigator]: New publisher discovered on topic 'hazard_detection', offering incompatible QoS. No messages will be received from it. Last incompatible policy: RELIABILITY
[INFO] [1768831997.946911406] [basic_navigator]: Publishing Initial Pose


In [4]:
# Wait for Nav2
navigator.waitUntilNav2Active()

[INFO] [1768832008.007509676] [basic_navigator]: Nav2 is ready for use!


## Set waypoints 1 and execute

In [ ]:
## Add waypoints
point_A = [-3.4, -1.5]
point_A_1 = [-2.14, -4.31]
goal_pose = []

goal_pose.append(navigator.getPoseStamped(point_A, TurtleBot4Directions.EAST))
# goal_pose.append(navigator.getPoseStamped(point_A_1, TurtleBot4Directions.EAST))
goal_pose.append(initial_pose)

In [ ]:
# Start getting out of dock
if navigator.getDockedStatus():
    navigator.info('Undocking before chasing pose')
    navigator.undock()
    
navigator.startFollowWaypoints(goal_pose)
navigator.dock()

## Set waypoints 2

<img src="../maps/test-arena-2.png" alt="isolated" width="400"/>

In [ ]:
## Add waypoints
point_B = [1.84, -6.38]
point_C = [2.46, -2.12]

# goal_pose = []

goal_pose.append(navigator.getPoseStamped(point_B, TurtleBot4Directions.EAST))
goal_pose.append(navigator.getPoseStamped(point_A_1, TurtleBot4Directions.EAST))
goal_pose.append(navigator.getPoseStamped(point_C, TurtleBot4Directions.SOUTH))
goal_pose.append(initial_pose)

In [ ]:
# Start getting out of dock
if navigator.getDockedStatus():
    navigator.info('Undocking before chasing pose')
    navigator.undock()
    
navigator.startFollowWaypoints(goal_pose)
navigator.dock()

## Dock Robot

In [ ]:
# Finished navigating, dock
navigator.dock()

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image)
plt.axis('off')

## Cancel Tasks

In [ ]:
navigator.cancelTask()

# goal_pose = [initial_pose]
# navigator.startFollowWaypoints(goal_pose)
# navigator.dock()

In [ ]:
rclpy.shutdown()

# Continuous Navigation

In [1]:
import rclpy

from turtlebot4_navigation.turtlebot4_navigator import TurtleBot4Directions, TurtleBot4Navigator

In [2]:
rclpy.init()
navigator = TurtleBot4Navigator()

In [3]:
# Start on dock
if not navigator.getDockedStatus():
    navigator.info('Docking before intialising pose')
    navigator.dock()
    
# Set initial pose
initial_pose = navigator.getPoseStamped([0.0, 0.0], TurtleBot4Directions.NORTH)

# initial_pose = navigator.getPoseStamped([-0.1, -0.1], 2.5)
navigator.setInitialPose(initial_pose)

[INFO] [1771918888.114217907] [basic_navigator]: Publishing Initial Pose


In [4]:
navigator.waitUntilNav2Active()

[INFO] [1770627177.173096381] [basic_navigator]: Setting initial pose
[INFO] [1770627177.174592405] [basic_navigator]: Publishing Initial Pose
[INFO] [1770627177.175777916] [basic_navigator]: Waiting for amcl_pose to be received
[INFO] [1770627177.176919814] [basic_navigator]: Setting initial pose
[INFO] [1770627177.177800026] [basic_navigator]: Publishing Initial Pose
[INFO] [1770627177.178554202] [basic_navigator]: Waiting for amcl_pose to be received
[INFO] [1770627177.180268825] [basic_navigator]: Setting initial pose
[INFO] [1770627177.181360498] [basic_navigator]: Publishing Initial Pose
[INFO] [1770627177.183832568] [basic_navigator]: Waiting for amcl_pose to be received
[INFO] [1770627179.190972452] [basic_navigator]: Nav2 is ready for use!


In [5]:
initial_pose

geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1770627175, nanosec=92522313), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.0, y=0.0, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)))

In [7]:
# point_A = [0.05, 3.]
point_A = [-4.5, -1.]
point_B = [0.05, 3.]
goal_pose = navigator.getPoseStamped(point_A, TurtleBot4Directions.EAST)

In [8]:
## Get Path and visualize on Rviz
path = navigator.getPath(initial_pose, goal_pose)

[INFO] [1770711263.338222974] [basic_navigator]: Getting path...


In [9]:
smoothed_path = navigator.smoothPath(path)

[INFO] [1770711264.011932029] [basic_navigator]: Smoothing path...


In [12]:
smoothed_path.poses

[geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1770711263, nanosec=339873152), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=-0.019999900162219753, y=7.078051567077637e-08, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.045465863476804994, w=0.9989658929404489))),
 geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1770711263, nanosec=339873152), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=-0.04607877569541914, y=-0.0023787086931131114, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.04076336077836933, w=0.9991688287862329))),
 geometry_msgs.msg.PoseStamped(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1770711263, nanosec=339873152), frame_id='map'), pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=-0.0720587729616988, y=-0.0045020687917916, z=0.0), orien

In [16]:
if navigator.getDockedStatus():
    navigator.info('Docking before intialising pose')
    navigator.undock()
    
# navigator.goToPose(goal_pose)
navigator.goToPose(initial_pose)

# navigator.followPath(smoothed_path)
# while not navigator.isTaskComplete():
#     feedback = navigator.getFeedback()
#     print(feedback)

#     if feedback.distance_to_goal < 3:
#         update goal
#         point_B = [0.05, 3.]
#         updated_goal_pose = navigator.getPoseStamped(point_A, TurtleBot4Directions.EAST)
#         updated_path = navigator.getPath(initial_pose, updated_goal_pose)
#         navigator.followPath(updated_path)
        
    # if feedback.navigation_duration > 600:
    #     navigator.cancelTask()

[INFO] [1770627370.527212466] [basic_navigator]: Navigating to goal: 0.0 0.0...


True

In [26]:
navigator.followObjectByFrame('person [1]')

AttributeError: 'TurtleBot4Navigator' object has no attribute 'followObjectByFrame'

In [24]:
navigator.followPath(smoothed_path)
while not navigator.isTaskComplete():
    feedback = navigator.getFeedback()
    print(feedback)

    if feedback.distance_to_goal < 3:
        navigator.cancelTask()
        
point_B = [0.05, 3.]
updated_goal_pose = navigator.getPoseStamped(point_B, TurtleBot4Directions.EAST)
updated_path = navigator.getPath(initial_pose, updated_goal_pose)
navigator.followPath(updated_path)
while not navigator.isTaskComplete():
    feedback = navigator.getFeedback()
    print(feedback)

[INFO] [1770628575.533378047] [basic_navigator]: Executing path...


nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.027368420735001564)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.027368420735001564)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.027368420735001564)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.027368420735001564)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=5.133058547973633, speed=0.05473684147

[INFO] [1770628600.725569788] [basic_navigator]: Canceling current task.
[INFO] [1770628600.741902018] [basic_navigator]: Getting path...
[INFO] [1770628600.778742337] [basic_navigator]: Executing path...


nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.2907114028930664, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.0)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.05473684147000313)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.027368420735001564)
nav2_msgs.action.FollowPath_Feedback(distance_to_goal=3.31571102142334, speed=0.027368420735001564)
nav2_msgs.action.FollowPath_Feedbac

In [19]:
result = navigator.getResult()
result

<TaskResult.CANCELED: 2>

In [28]:
navigator.goToPose(initial_pose)

[INFO] [1770629223.924316078] [basic_navigator]: Navigating to goal: 0.0 0.0...


True

In [5]:
navigator.dock()

[INFO] [1771919830.229519603] [basic_navigator]: Docking...
[INFO] [1771919846.153687706] [basic_navigator]: Dock succeeded


In [6]:
navigator.undock()

[INFO] [1771920273.828224813] [basic_navigator]: Undocking...
[INFO] [1771920279.471719487] [basic_navigator]: Undock succeeded


In [27]:
from turtlebot4_navigation.turtlebot4_object_follower import TurtleBot4ObjectFollower

ModuleNotFoundError: No module named 'turtlebot4_navigation.turtlebot4_object_follower'